# Milestone - Unit 9: the racing line

Unit 9 asked whether the centreline is the fast line, and what a new line does to the speed tuning.
`agents/unit-09.py` is Unit 8's file driving the **smoothest line**: each waypoint moves n_i metres along its left unit normal, the offsets are a periodic uniform cubic B-spline of `LINE_CONTROL` (one control value per `LINE_STEP` = 10 waypoints), and the control values minimise the sum of squared second differences of the line's points, subject to |n_i| ≤ w_i − 2.5 m at every waypoint and n = 0 on the 30 waypoints either side of the start.
The line is solved offline (QR, then `scipy.optimize.lsq_linear(method="bvls")` on R c = Qᵀb) and shipped as constants; the agent measures, plans and pursues on its own line points, and plans with `MU_FRACTION = 0.54` of Unit 8's measured grip, re-tuned for the line by the margin rule.

The notebook asserts the unit's gates (plan `roar-008`) on the trimmed subset below; a routed racekit test (`tests/test_racekit_unit08_09_premise.py`) races every multi-seed figure in full, on every track:

- **U9.1** the line is valid (race-free): on `test-circuit`, |n_i| ≤ w_i − 2.5 m + 1e-9 at every waypoint, |n_i| ≤ 1e-9 on the 30 waypoints either side of the start, Σκ²Δs 0.2880 → 0.2326 (±0.002), the lap 2661.7 → 2636.2 m and the tightest radius 18.0 → 15.2 m; the shipped `LINE_CONTROL` re-solves to within 1e-6 m and the agent rebuilds it; on `monza-like` the same validity facts, 0.2421 → 0.1428, 5714.5 → 5684.7 m and 25.0 → 28.3 m;
- **U9.2** the same f: on `test-circuit` at 0.54, seeds 0–4, the line is clean at 279.60 s and the centreline at 304.70 s, a noise-aware win; on `monza-like` at 0.60, seed 0, the line 473.45 s, local widening 487.55 s and the centreline 500.70 s, all clean;
- **U9.3** a new line breaks the tuning: on `test-circuit-seed4`, seed 0, the centreline at 0.58 (Unit 8's edge) is clean and the line at 0.58 is not;
- **U9.4** the family and the transfer: with the line, 0.56 is clean on `test-circuit-seed4` (the track where 0.58 fails) at 273.45 s, seeds 0–4; 0.54 is clean on the unseen `test-circuit-seed2026` at 282.05 s, seeds 0–4, and 0.58 is not clean there (seed 0);
- **U9.5** local widening (60-waypoint windows) is not clean on `test-circuit` at 0.50 (seed 0);
- **U9.6** the reference: `agents/unit-09.py` on `test-circuit`, clean at 279.60 s over seeds 0–4 (U9.2's races); `check_submission` passes; with `LINE_CONTROL = None` it races exactly like `agents/unit-08.py` at 0.54 (seed 0, identical telemetry).

Every race uses racekit's carla-like car, `carla_like(noise=True)`, seeded by `run_race`, with `max_seconds` 450 on the `test-circuit` family and 900 on `monza-like`.
Tolerances (the plan's Gates header): medians and single-seed times within one tick (±0.05 s) of the pinned median; outcomes exact; "not clean" is asserted, never a collision count.
A race is **clean** when it finished, with 0 collisions, and its max |cross-track error from its own line| (the centreline offset minus n at the car's waypoint (progress + 1) mod n, over the driving rows) is at most 2 m.
The per-track constants (`MU_G`, Unit 8's measured grip, and `LINE_CONTROL`) are written in with `write_variant`, as Checkpoint A's settings were.
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
The line solver is Unit 9's (§3 of the lesson): QR, then bounded least squares on the m × m problem.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("..") / "agents"
UNIT_08, UNIT_09 = AGENTS / "unit-08.py", AGENTS / "unit-09.py"
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9
CLEAN_ERROR = 2.0  # m
STEP, MARGIN, PIN = 10, 2.5, 30
LOCAL_RADIUS, LOCAL_WINDOW = 60.0, 60
MEASURED_MU_G = {  # Unit 8's grip probes, mu * g per section (m/s^2, rounded to 0.01)
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
    "monza-like": (5.38, 8.96, 9.12, 9.14, 9.70, 7.97),
}
workdir = Path(mkdtemp(prefix="roar-milestone-09-"))


def menger(x, y):
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def spline_basis(n, step=STEP):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, local=False):
    # The smoothest line's control values (QR + bvls); local=True opens only windows around the tight points.
    x, y = track.x, track.y
    n = len(x)
    nx, ny = normals(x, y)
    basis = spline_basis(n)
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    index = np.arange(n)
    fixed = (index < PIN) | (index >= n - PIN)
    if local:
        near = np.zeros(n, dtype=bool)
        for i in np.nonzero(np.abs(menger(x, y)) > 1.0 / LOCAL_RADIUS)[0]:
            near[np.arange(i - LOCAL_WINDOW, i + LOCAL_WINDOW + 1) % n] = True
        fixed = fixed | ~near
    high = np.zeros(basis.shape[1])
    for j in range(basis.shape[1]):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(fixed[support]) > 0 else np.min(track.half_width[support] - MARGIN)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


def line_summary(x, y):
    kappa = menger(x, y)
    ds = np.hypot(np.roll(x, -1) - x, np.roll(y, -1) - y)
    return float(np.sum(kappa ** 2 * ds)), float(ds.sum()), float(1.0 / np.max(np.abs(kappa)))


def check_valid(track, control):
    # U9.1's validity facts; returns (the centreline's numbers, the line's numbers).
    n = len(track)
    offsets = spline_basis(n) @ control
    assert len(control) == n // STEP
    assert np.max(np.abs(offsets) - (track.half_width - MARGIN)) <= 1e-9
    assert np.max(np.abs(offsets[:PIN])) <= 1e-9 and np.max(np.abs(offsets[n - PIN:])) <= 1e-9
    nx, ny = normals(track.x, track.y)
    return line_summary(track.x, track.y), line_summary(track.x + offsets * nx, track.y + offsets * ny)


def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def cap_for(track_name):
    return 900.0 if track_name == "monza-like" else 450.0


def agent_variant(agent, name, track_name, control=None, **constants):
    # A reference with the track's measured grip and, for unit-09.py, the track's line (test-circuit: as shipped).
    if track_name != "test-circuit":
        constants["MU_G"] = MEASURED_MU_G[track_name]
        if agent == UNIT_09:
            constants["LINE_CONTROL"] = tuple(float(value) for value in control)
    if not constants:
        return agent
    return write_variant(agent, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track_name, seed=0):
    return run_race(source, track_name, car=carla_like(noise=True), seed=seed, max_seconds=cap_for(track_name))


def seeds_race(source, track_name):
    return repeat_races(source, track_name, n=len(SEEDS), seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                        max_seconds=cap_for(track_name))

## 2. U9.1: the line is valid, and the shipped one re-solves

`test-circuit`'s line, solved here, against the 133 values `agents/unit-09.py` ships and the offsets its `__init__` rebuilds from them; then `monza-like`'s line (about 10 s to solve), which U9.2 races.

In [ ]:
track = load_track("test-circuit")
n = len(track)
control = solve_line(track)
centre, line = check_valid(track, control)
print(f"test-circuit: sum kappa^2 ds {centre[0]:.4f} -> {line[0]:.4f}, lap {centre[1]:.1f} -> {line[1]:.1f} m, "
      f"tightest radius {centre[2]:.1f} -> {line[2]:.1f} m")
assert abs(centre[0] - 0.2880) <= 5e-5 and abs(line[0] - 0.2326) <= 0.002
assert abs(centre[1] - 2661.7) <= 0.05 and abs(line[1] - 2636.2) <= 0.05
assert abs(centre[2] - 18.0) <= 0.05 and abs(line[2] - 15.2) <= 0.05

with shim_installed():
    Unit09, _ = load_submission(UNIT_09)
agent = Unit09(track.waypoints, None)
offsets = spline_basis(n) @ control
gap = np.max(np.abs(np.array(agent.offsets) - offsets))
nx, ny = normals(track.x, track.y)
print(f"the shipped line against the re-solved one: {gap:.0e} m")
assert gap <= 1e-6
assert np.max(np.abs(np.array(agent.xs) - (track.x + offsets * nx))) <= 1e-6
assert np.max(np.abs(np.array(agent.ys) - (track.y + offsets * ny))) <= 1e-6

monza = load_track("monza-like")
monza_control = solve_line(monza)
monza_offsets = spline_basis(len(monza)) @ monza_control
centre, line = check_valid(monza, monza_control)
print(f"monza-like: sum kappa^2 ds {centre[0]:.4f} -> {line[0]:.4f}, lap {centre[1]:.1f} -> {line[1]:.1f} m, "
      f"tightest radius {centre[2]:.1f} -> {line[2]:.1f} m")
assert abs(centre[0] - 0.2421) <= 5e-5 and abs(line[0] - 0.1428) <= 0.002
assert abs(centre[1] - 5714.5) <= 0.05 and abs(line[1] - 5684.7) <= 0.05
assert abs(centre[2] - 25.0) <= 0.05 and abs(line[2] - 28.3) <= 0.05

## 3. U9.6 and U9.2: the reference, and the same f

The reference on `test-circuit`, seeds 0–4, against `agents/unit-08.py` (the centreline) at the same f, 0.54; then `check_submission` and the equivalence: with `LINE_CONTROL = None`, Unit 9's file is Unit 8's at 0.54, tick for tick.

In [ ]:
reference = seeds_race(UNIT_09, "test-circuit")
centre_054 = seeds_race(write_variant(UNIT_08, workdir / "centre-0.54.py", {"MU_FRACTION": "0.54"}), "test-circuit")
for name, result, line_offsets in (("the line (the reference)", reference, offsets),
                                   ("the centreline", centre_054, np.zeros(n))):
    print(f"test-circuit, f = 0.54, {name}: median {result.median:.2f} s (spread {result.spread:.2f}), "
          f"max error from its line {max(line_error(r, line_offsets) for r in result.results):.2f} m")
assert all(clean(r, offsets) for r in reference.results) and abs(reference.median - 279.60) <= TICK
assert all(clean(r, np.zeros(n)) for r in centre_054.results) and abs(centre_054.median - 304.70) <= TICK
assert noise_aware_better(reference.times, centre_054.times)

report = check_submission(UNIT_09)
print(report)
assert report.ok, report.findings

no_line = race(write_variant(UNIT_09, workdir / "no-line.py", {"LINE_CONTROL": "None"}), "test-circuit")
unit_08 = centre_054.results[0]  # agents/unit-08.py at 0.54, seed 0
same = all(np.sum(no_line.telemetry[column] != unit_08.telemetry[column]) == 0 for column in no_line.telemetry.dtype.names)
print(f"LINE_CONTROL = None: {no_line.total_time:.2f} s, Unit 8 at 0.54: {unit_08.total_time:.2f} s, identical: {same}")
assert same and no_line.total_time == unit_08.total_time and no_line.collisions == unit_08.collisions

`monza-like` at f = 0.60, seed 0: the global line, local widening (60-waypoint windows around every point tighter than 60 m) and the centreline.

In [ ]:
monza_local = solve_line(monza, local=True)
monza_local_offsets = spline_basis(len(monza)) @ monza_local
check_valid(monza, monza_local)
monza_line = race(agent_variant(UNIT_09, "monza-line-0.60", "monza-like", monza_control, MU_FRACTION=0.60), "monza-like")
monza_widened = race(agent_variant(UNIT_09, "monza-local-0.60", "monza-like", monza_local, MU_FRACTION=0.60),
                     "monza-like")
monza_centre = race(agent_variant(UNIT_08, "monza-centre-0.60", "monza-like", MU_FRACTION=0.60), "monza-like")
for name, result, pinned, line_offsets in (("the line", monza_line, 473.45, monza_offsets),
                                           ("local widening", monza_widened, 487.55, monza_local_offsets),
                                           ("the centreline", monza_centre, 500.70, np.zeros(len(monza)))):
    print(f"monza-like, f = 0.60, seed 0, {name}: {result.total_time:.2f} s, clean {clean(result, line_offsets)}")
    assert clean(result, line_offsets) and abs(result.total_time - pinned) <= TICK

## 4. U9.3 and U9.4: a new line breaks the tuning, and the re-tune transfers

On `test-circuit-seed4`, seed 0, Unit 8's edge value 0.58 on the centreline (clean) and on the track's own line (not clean); then the line at Unit 8's frozen 0.56 on seeds 0–4, clean but now at the edge.

In [ ]:
seed4 = load_track("test-circuit-seed4")
seed4_control = solve_line(seed4)
check_valid(seed4, seed4_control)
seed4_offsets = spline_basis(len(seed4)) @ seed4_control
centre_058 = race(agent_variant(UNIT_08, "seed4-centre-0.58", "test-circuit-seed4", MU_FRACTION=0.58), "test-circuit-seed4")
line_058 = race(agent_variant(UNIT_09, "seed4-line-0.58", "test-circuit-seed4", seed4_control, MU_FRACTION=0.58),
                "test-circuit-seed4")
print(f"test-circuit-seed4, f = 0.58, seed 0: the centreline clean {clean(centre_058, np.zeros(len(seed4)))}, "
      f"the line clean {clean(line_058, seed4_offsets)}")
assert clean(centre_058, np.zeros(len(seed4)))
assert not clean(line_058, seed4_offsets)  # U9.3

line_056 = seeds_race(agent_variant(UNIT_09, "seed4-line-0.56", "test-circuit-seed4", seed4_control, MU_FRACTION=0.56),
                      "test-circuit-seed4")
print(f"test-circuit-seed4, the line, f = 0.56: median {line_056.median:.2f} s (spread {line_056.spread:.2f})")
assert all(clean(r, seed4_offsets) for r in line_056.results) and abs(line_056.median - 273.45) <= TICK  # U9.4

The transfer: the re-tuned 0.54 on the unseen `test-circuit-seed2026`, seeds 0–4, and 0.58 there (seed 0).

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
check_valid(seed2026, seed2026_control)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
frozen = seeds_race(agent_variant(UNIT_09, "seed2026-line-0.54", "test-circuit-seed2026", seed2026_control),
                    "test-circuit-seed2026")
edge = race(agent_variant(UNIT_09, "seed2026-line-0.58", "test-circuit-seed2026", seed2026_control, MU_FRACTION=0.58),
            "test-circuit-seed2026")
print(f"test-circuit-seed2026, the line, f = 0.54: median {frozen.median:.2f} s (spread {frozen.spread:.2f})")
print(f"test-circuit-seed2026, the line, f = 0.58, seed 0: clean {clean(edge, seed2026_offsets)}, "
      f"{line_error(edge, seed2026_offsets):.2f} m from its line")
assert all(clean(r, seed2026_offsets) for r in frozen.results) and abs(frozen.median - 282.05) <= TICK  # U9.4
assert not clean(edge, seed2026_offsets)

## 5. U9.5: local widening loses on `test-circuit`

Local widening with 60-waypoint windows at f = 0.50, seed 0.

In [ ]:
local_control = solve_line(track, local=True)
check_valid(track, local_control)
local_offsets = spline_basis(n) @ local_control
local_race = race(write_variant(UNIT_09, workdir / "local-0.50.py",
                                {"LINE_CONTROL": repr(tuple(float(value) for value in local_control)),
                                 "MU_FRACTION": "0.5"}), "test-circuit")
print(f"test-circuit, local widening, f = 0.50, seed 0: clean {clean(local_race, local_offsets)}")
assert not clean(local_race, local_offsets)  # U9.5

## Unit 9, pinned

| medians, seeds 0–4, carla-like car | the centreline | the racing line |
|---|---|---|
| `test-circuit`, f = 0.54 | 304.70 s | **279.60 s** (the reference) |
| `monza-like`, f = 0.60 (local widening 487.55 s) | 500.70 s | 473.45 s |
| `test-circuit-seed2026` (unseen) | 303.30 s (Unit 8, f = 0.56) | **282.05 s** (f = 0.54) |

| the family edge of `MU_FRACTION` | the centreline | the racing line |
|---|---|---|
| clean on all five training tracks up to | 0.58 | 0.56 |
| frozen by the margin rule | 0.56 | **0.54** |

The submission ships `test-circuit`'s line and grip; on another track both are written in with `write_variant`.